# 07 — Local demo (Gradio)

Classify the **whole uploaded image** (single-fish or tight framing works best; crop first if the subject is small).

Pick **one** `TAX_CKPT` below (SSL, FedAvg, few-shot, or compressed from notebook `08`).

Source: https://www.gradio.app/guides/quickstart

In [ ]:
from pathlib import Path
import sys

import torch
from PIL import Image, ImageDraw

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)


In [ ]:
import gradio as gr
from torchvision.transforms import functional as F

# --- Taxonomy: uncomment exactly ONE checkpoint ---
TAX_CKPT = (ROOT / 'outputs' / 'taxonomy_ssl' / 'taxonomy_ssl_resnet50.pt').resolve()  # notebook 03
# TAX_CKPT = (ROOT / 'outputs' / 'federated_fedavg' / 'fedavg_global_model.pt').resolve()  # notebook 05
# TAX_CKPT = (ROOT / 'outputs' / 'fewshot_taxonomy' / 'k10_seed0' / 'model.pt').resolve()  # notebook 04 — edit folder
# TAX_CKPT = (ROOT / 'outputs' / 'taxonomy_ssl' / 'taxonomy_ssl_resnet50_compressed.huff.pt').resolve()  # notebook 08 (CPU)

print('TAX_CKPT:', TAX_CKPT)
print('  exists:', TAX_CKPT.exists())


In [ ]:
from fish_ai.compress.pipeline import load_taxonomy_checkpoint_auto, load_taxonomy_for_inference

taxonomy_device = device


def load_taxonomy():
    global taxonomy_device
    ckpt = load_taxonomy_checkpoint_auto(TAX_CKPT, map_location='cpu')
    maps = ckpt['maps']
    inv = {lvl: {i: s for s, i in maps[lvl].items()} for lvl in maps.keys()}
    model = load_taxonomy_for_inference(ckpt)
    comp = ckpt.get('compression') or {}
    taxonomy_device = torch.device('cpu') if comp.get('quantized') else device
    model.to(taxonomy_device).eval()
    return model, inv


if TAX_CKPT.exists():
    tax_model, inv_maps = load_taxonomy()
else:
    tax_model, inv_maps = None, None
    taxonomy_device = device

tax_model is not None


In [ ]:
def softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    return torch.softmax(x, dim=dim)


def classify_image(img: Image.Image):
    x = F.to_tensor(img.resize((224, 224))).unsqueeze(0).to(taxonomy_device)
    out = tax_model(x)
    probs_s = softmax(out['species'], dim=1)[0]
    probs_g = softmax(out['genus'], dim=1)[0]
    probs_f = softmax(out['family'], dim=1)[0]
    s_id = int(torch.argmax(probs_s).item())
    g_id = int(torch.argmax(probs_g).item())
    f_id = int(torch.argmax(probs_f).item())
    return {
        'family': inv_maps['family'][f_id],
        'genus': inv_maps['genus'][g_id],
        'species': inv_maps['species'][s_id],
        'species_conf': float(probs_s[s_id].item()),
    }


def predict(image: Image.Image, cls_thresh: float = 0.5):
    if tax_model is None or inv_maps is None:
        msg = 'Set TAX_CKPT to a trained .pt (notebooks 03 / 04 / 05 / 08).'
        return image, [{'error': msg}]
    if image is None:
        return None, [{'error': 'No image uploaded.'}]
    img = image.convert('RGB')
    pred = classify_image(img)
    label = pred['species']
    if pred['species_conf'] < cls_thresh:
        label = 'Unknown'
    draw = ImageDraw.Draw(img)
    text = f"{label}  cls={pred['species_conf']:.2f}"
    draw.rectangle([0, 0, img.width - 1, img.height - 1], outline='teal', width=4)
    draw.text((8, 8), text, fill='white', stroke_width=2, stroke_fill='black')
    return img, [
        {
            'family': pred['family'],
            'genus': pred['genus'],
            'species': pred['species'],
            'species_conf': pred['species_conf'],
            'final_label': label,
        }
    ]


In [ ]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type='pil', label='Upload image'),
        gr.Slider(0, 1, value=0.5, step=0.05, label='Unknown threshold (species confidence)'),
    ],
    outputs=[
        gr.Image(type='pil', label='Output'),
        gr.JSON(label='Results'),
    ],
    title='Fish AI — taxonomy (full image)',
    description='Upload one fish image. Species becomes Unknown if confidence is below the threshold.',
)

demo.launch()
